In [1]:
import pandas as pd
import numpy as np
import krippendorff
from collections import Counter

In [2]:
def load_all_data(directory='../data'):
    """Loads master and labeller files into a single standardized format."""
    print("Loading data...")
    master_df = pd.read_csv(f'{directory}/label_master.csv')
    
    all_labels = []
    for i in range(1, 7):
        df = pd.read_csv(f'{directory}/label_{i}.csv')
        df['labeller_id'] = i  
        all_labels.append(df)
        
    labels_df = pd.concat(all_labels, ignore_index=True)
    merged_df = pd.merge(labels_df, master_df[['entry_id', 'Set_Type']], on='entry_id', how='left')
    
    print(f"Data loaded successfully. {len(labels_df)} total labels found.")
    return merged_df, master_df

merged_data, master_data = load_all_data()

Loading data...
Data loaded successfully. 2361 total labels found.


In [3]:
def generate_progress_report(merged_df, master_df):
    """Generates a progress report looking for fully completed labels."""
    print("\n" + "="*60)
    print("=== Labeller Progress & Statistics ===")
    print("="*60)
    
    total_overlap = len(master_df[master_df['Set_Type'] == 'Overlap'])
    feas_cols = ['feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']
    
    stats = []
    for i in range(1, 7):
        assigned_indiv = len(master_df[(master_df['Set_Type'] == 'Individual') & (master_df['Assignee'] == i)])
        labeller_data = merged_df[merged_df['labeller_id'] == i]
        
        # A post is complete if it's 'No', OR if it's 'Yes' AND all feas scores are filled
        is_no = labeller_data['is_relevant'] == 'No'
        is_yes_and_filled = (labeller_data['is_relevant'] == 'Yes') & (labeller_data[feas_cols].notna().all(axis=1))
        completed_mask = is_no | is_yes_and_filled
        
        completed_overlap = (completed_mask & (labeller_data['Set_Type'] == 'Overlap')).sum()
        completed_indiv = (completed_mask & (labeller_data['Set_Type'] == 'Individual')).sum()
        
        irrelevant_count = is_no.sum()
        
        total_assigned = total_overlap + assigned_indiv
        total_completed = completed_overlap + completed_indiv
        progress_pct = (total_completed / total_assigned * 100) if total_assigned > 0 else 0
        
        stats.append({
            'Labeller': f"Labeller {i}",
            'Overlap Done': f"{completed_overlap}/{total_overlap}",
            'Individual Done': f"{completed_indiv}/{assigned_indiv}",
            'Progress (%)': f"{progress_pct:.1f}%",
            'Irrelevant Flags': irrelevant_count
        })
        
    stats_df = pd.DataFrame(stats)
    print(stats_df.to_string(index=False))
    
    stats_df.to_csv('labeller_progress_report.csv', index=False)
    print("\nSaved progress to 'labeller_progress_report.csv'")
    return stats_df

progress_df = generate_progress_report(merged_data, master_data)


=== Labeller Progress & Statistics ===
  Labeller Overlap Done Individual Done Progress (%)  Irrelevant Flags
Labeller 1      236/236         156/157        99.7%               220
Labeller 2      235/236         157/158        99.5%               172
Labeller 3      236/236         158/158       100.0%               293
Labeller 4       39/236         158/158        50.0%               108
Labeller 5      236/236         156/157        99.7%               189
Labeller 6      236/236          25/157        66.4%               128

Saved progress to 'labeller_progress_report.csv'


In [4]:
def run_irr_analysis(merged_df, master_df):
    """Calculates Krippendorff's Alpha on the Overlap set."""
    print("\n" + "="*45)
    print("=== Inter-Rater Reliability Analysis ===")
    print("="*45)
    
    variables = ['is_relevant', 'feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']
    feas_cols = ['feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']
    
    # Get all unique overlap entry IDs in a fixed order
    overlap_entries = master_df[master_df['Set_Type'] == 'Overlap']['entry_id'].tolist()
    
    rater_matrices = {var: [] for var in variables}
    
    # Build rater matrices ensuring exact alignment of entries
    for rater_id in range(1, 7):
        rater_row = {var: [] for var in variables}
        rater_data = merged_df[(merged_df['labeller_id'] == rater_id) & (merged_df['entry_id'].isin(overlap_entries))]
        rater_dict = rater_data.set_index('entry_id').to_dict('index')
        
        for eid in overlap_entries:
            if eid in rater_dict:
                row = rater_dict[eid]
                # Convert relevance to binary
                is_rel = 1 if row.get('is_relevant') == 'Yes' else (0 if row.get('is_relevant') == 'No' else np.nan)
                rater_row['is_relevant'].append(is_rel)
                
                # Nullify feasibility scores if marked irrelevant
                for col in feas_cols:
                    if is_rel == 0:
                        rater_row[col].append(np.nan)
                    else:
                        rater_row[col].append(row.get(col, np.nan))
            else:
                # Missing data if rater hasn't reached this entry
                for var in variables:
                    rater_row[var].append(np.nan)
                    
        for var in variables:
            rater_matrices[var].append(rater_row[var])
            
    # Calculate Alpha
    for var in variables:
        matrix = np.array(rater_matrices[var], dtype=float)
        valid_items_mask = np.sum(~np.isnan(matrix), axis=0) >= 2
        filtered_matrix = matrix[:, valid_items_mask]
        
        if filtered_matrix.shape[1] == 0:
            print(f"{var}: Not enough overlapping data yet.")
            continue
            
        try:
            level = 'nominal' if var == 'is_relevant' else 'ordinal'
            alpha = krippendorff.alpha(reliability_data=filtered_matrix, level_of_measurement=level)
            print(f"{var:<15} Alpha: {alpha:>.3f}  (Items: {filtered_matrix.shape[1]})")
        except ValueError as e:
            print(f"{var}: Could not compute ({e})")

run_irr_analysis(merged_data, master_data)


=== Inter-Rater Reliability Analysis ===
is_relevant     Alpha: 0.363  (Items: 236)
feas_pa         Alpha: 0.544  (Items: 145)
feas_ka         Alpha: 0.374  (Items: 145)
feas_cr         Alpha: 0.375  (Items: 145)
feas_sr         Alpha: 0.376  (Items: 145)


In [5]:
def calculate_majority_vote(series):
    """Helper to get majority vote for 'Yes'/'No', defaulting to 'Yes' on ties"""
    series = series.dropna()
    if len(series) == 0:
        return np.nan
    counts = Counter(series)
    if counts['Yes'] == counts['No']:
        return 'Yes'
    return counts.most_common(1)[0][0]

def create_model_dataset(merged_df, master_df, directory='../data'):
    """Calibrates biases, combines data, and formats for model training."""
    print("\n" + "="*60)
    print("=== Dataset Calibration & Compilation ===")
    print("="*60)
    
    feas_cols = ['feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']
    overlap_df = merged_df[merged_df['Set_Type'] == 'Overlap']
    
    consensus_overlap = pd.DataFrame()
    consensus_overlap['is_relevant'] = overlap_df.groupby('entry_id')['is_relevant'].agg(calculate_majority_vote)
    for col in feas_cols:
        consensus_overlap[col] = overlap_df.groupby('entry_id')[col].median().round()
        
    consensus_overlap = consensus_overlap.reset_index()
    
    bias_check = pd.merge(overlap_df, consensus_overlap, on='entry_id', suffixes=('_rater', '_consensus'))
    rater_biases = {i: {col: 0.0 for col in feas_cols} for i in range(1, 7)}
    
    print("Average Labeller Biases (Positive = scores higher than group median):")
    for i in range(1, 7):
        rater_data = bias_check[bias_check['labeller_id'] == i]
        for col in feas_cols:
            diffs = rater_data[f'{col}_rater'] - rater_data[f'{col}_consensus']
            mean_bias = diffs.mean()
            if not pd.isna(mean_bias):
                rater_biases[i][col] = mean_bias
                
    bias_df = pd.DataFrame(rater_biases).T
    bias_df.index.name = 'Labeller'
    print(bias_df.round(2))
                
    individual_df = merged_df[merged_df['Set_Type'] == 'Individual'].copy()
    
    for idx, row in individual_df.iterrows():
        rater = row['labeller_id']
        if row['is_relevant'] == 'Yes':
            for col in feas_cols:
                if not pd.isna(row[col]):
                    calibrated_score = row[col] - rater_biases[rater][col]
                    individual_df.at[idx, col] = np.clip(np.round(calibrated_score), 1, 5)
                    
    individual_final = individual_df[['entry_id', 'is_relevant'] + feas_cols]
    final_dataset = pd.concat([consensus_overlap, individual_final], ignore_index=True)
    
    # 1. Keep only relevant posts
    final_dataset = final_dataset[final_dataset['is_relevant'] == 'Yes'].copy()
    
    # 2. Drop rows where any of the 1-5 feasibility scores are missing 
    # (Handles the edge case of 'Yes' but left unlabelled)
    final_dataset = final_dataset.dropna(subset=feas_cols)
    
    # 3. Merge with master to get text AND the Assignee column
    final_dataset = pd.merge(final_dataset, master_df[['entry_id', 'title', 'selftext', 'Assignee']], on='entry_id', how='left')
    
    # 4. Format Assignee: fill NaNs with 'Overlap' and format actual numbers as strings
    final_dataset['Assignee'] = final_dataset['Assignee'].apply(
        lambda x: 'Overlap' if pd.isna(x) else str(int(x))
    )
    
    # 5. Reorder columns (this naturally drops 'is_relevant' and 'entry_id')
    final_cols = ['title', 'selftext', 'Assignee'] + feas_cols
    final_dataset = final_dataset[final_cols]
    
    output_file = f'{directory}/model_training_data.csv'
    final_dataset.to_csv(output_file, index=False)
    print(f"\nSuccess! Calibrated and filtered data saved to '{output_file}'")
    print(f"Final dataset shape: {final_dataset.shape} (Rows: Relevant Posts, Columns: Features & Targets)")
    
    return final_dataset

In [6]:
final_training_data = create_model_dataset(merged_data, master_data)

# Quick preview
print("\nPreview of training data:")
print(final_training_data.head(3))

print(f"Training data size: {len(final_training_data)}")


=== Dataset Calibration & Compilation ===
Average Labeller Biases (Positive = scores higher than group median):
          feas_pa  feas_ka  feas_cr  feas_sr
Labeller                                    
1            0.19     0.34     0.04    -0.07
2           -0.46    -0.21    -0.30     0.16
3           -0.32     0.25     0.16     0.04
4           -0.11     0.21     0.58    -0.05
5            0.11     0.03     0.11     0.01
6            0.11     0.03     0.11     0.01

Success! Calibrated and filtered data saved to '../data/model_training_data.csv'
Final dataset shape: (476, 7) (Rows: Relevant Posts, Columns: Features & Targets)

Preview of training data:
                                               title  \
0  Accepted a Job, Relocated, and Then Got My Off...   
1               I Am a Fraud and ChatGPT Is My Brain   
2                       Wow interviews suck more now   

                                            selftext Assignee  feas_pa  \
0  In other words, they filled the ro